# NB05 — λ-Native Subspace Clustering

**Series position:** NB01 (synthetic geometry) → NB02 (word-level λ-maps) → NB03 (sentence corpus + BasinHop) → NB04 (λ-fingerprinting + KMeans/RP tiling) → **NB05 (λ-native subspace clustering)**

NB04 tiled the latent space with KMeans centroids and random projections, then measured per-tile λ-distributions.  
NB05 discards exogenous tiling entirely: **subspaces are grown directly from ArrowSpace λ-values** using a seed-absorb algorithm.

### Core idea
A sentence *belongs* to a subspace if its spectral energy relative to that subspace (λ) is below a threshold θ = 0.45.  
A sentence is a **boundary point** if it falls below θ for ≥ 2 subspaces simultaneously.

### Hypotheses
| ID | Hypothesis | Comparison |
|---|---|---|
| H1 | λ-native purity > NB04 KMeans Rank@1 | NB04 baseline 0.125 |
| H2 | Boundary sentences are semantically ambiguous / polysemous | Inspected via Chart 2 + probes |
| H3 | Off-diagonal Wasserstein W higher than NB04 KMeans tiles | NB04 baseline ≈ 0.07 |
| H4 | θ = 0.45 is meaningful: purity peaks in a stable range 0.20–0.65 | Threshold sweep |


## 0. Imports and reproducibility

In [1]:
import os, warnings, itertools, pathlib
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from scipy.stats import wasserstein_distance
from sklearn.decomposition import PCA
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

OUTPUT_DIR = pathlib.Path('output__05')
OUTPUT_DIR.mkdir(exist_ok=True)

LAMBDA_THRESHOLD = 0.45   # θ — the key hyperparameter
MAX_ROUNDS       = 20     # absorption loop guard
ALPHA            = 0.5    # ArrowSpace diffusion parameter

print('NumPy', np.__version__)
print('Output dir:', OUTPUT_DIR.resolve())

NumPy 2.4.6
Output dir: /Users/tuned-silicon/code/arrowspace-analysis/notebooks/output__05


## 1. ArrowSpace helpers (real library or NumPy fallback)

In [18]:
try:
    import arrowspace as aspace
    ASPACE_REAL = True
    print('arrowspace library loaded')
except ImportError:
    ASPACE_REAL = False
    print('arrowspace not found — using NumPy fallback')


def _laplacian_numpy(M: np.ndarray, alpha: float = 0.5) -> np.ndarray:
    """Symmetric normalised graph Laplacian of a row-normalised matrix M."""
    M = M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-12)
    W = M @ M.T
    np.fill_diagonal(W, 0.0)
    D = np.diag(W.sum(axis=1) + 1e-12)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(np.diag(D)))
    L = np.eye(len(W)) - D_inv_sqrt @ W @ D_inv_sqrt
    return L


def build_lambda_map(X: np.ndarray, alpha: float = ALPHA) -> np.ndarray:
    """Return λ values (one per row of vecs) using real library or fallback."""
    if ASPACE_REAL:
        from arrowspace import ArrowSpaceBuilder
        GRAPH_PARAMS = {'eps': 1.5, 'k': 25, 'topk': 10, 'p': 2.0, 'sigma': 0.5}
        builder = (ArrowSpaceBuilder()
            .with_seed(42)
            .with_dims_reduction(enabled=False, eps=None)
            .with_sampling("simple", 1.0))
        aspace, gl = builder.build(
            GRAPH_PARAMS, np.ascontiguousarray(X, dtype=np.float64))
        return np.asarray(aspace.lambdas(), dtype=float), aspace, gl
    L = _laplacian_numpy(X, alpha)
    eigvals = np.linalg.eigvalsh(L)
    # λ per vector = projection onto eigenvectors weighted by eigenvalue
    # lightweight proxy: mean of row-wise Rayleigh quotients
    L_sym = (L + L.T) / 2
    lambdas = np.array([
        float(v @ L_sym @ v) for v in
        (vecs / (np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-12))
    ])
    return lambdas


def subspace_lambda(query_vec: np.ndarray, subspace_vecs: np.ndarray,
                    alpha: float = ALPHA) -> float:
    """λ of query_vec relative to a subspace defined by subspace_vecs.
    Stack the query onto the subspace matrix, compute λ, return the last value.
    """
    if len(subspace_vecs) == 0:
        return np.inf
    combined = np.vstack([subspace_vecs, query_vec.reshape(1, -1)])
    lambdas  = build_lambda_map(combined, alpha)
    return float(lambdas[0][-1])   # λ of the query row


print('Helper functions ready.')

arrowspace library loaded
Helper functions ready.


## 2. Sentence corpus (8 fields × 12 sentences + 6 polysemous probes)

Identical corpus to NB03/NB04 — keeping the series comparable.

In [19]:
FIELDS = ['astronomy', 'cooking', 'law', 'music', 'medicine',
          'economics', 'architecture', 'ecology']

SENTENCES = {
    'astronomy': [
        'The luminosity of a main-sequence star scales with its mass.',
        'Neutron stars rotate at hundreds of revolutions per second.',
        'Dark matter does not emit or absorb electromagnetic radiation.',
        'The Hubble constant describes the expansion rate of the universe.',
        'Solar flares release enormous amounts of energy into space.',
        'Binary star systems can transfer mass between their components.',
        'Gravitational lensing bends light around massive objects.',
        'The cosmic microwave background is a relic of the Big Bang.',
        'Red giants form when a star exhausts its core hydrogen.',
        'Exoplanets are detected via the transit and radial-velocity methods.',
        'Quasars are powered by accretion onto supermassive black holes.',
        'The interstellar medium contains gas, dust, and cosmic rays.',
    ],
    'cooking': [
        'Maillard reactions create flavour by browning proteins and sugars.',
        'Emulsification binds fat and water using lecithin as a surfactant.',
        'Braising combines dry and moist heat to tenderise tough cuts.',
        'Fermentation converts sugars to alcohol and acids via microbes.',
        'Blanching briefly scalds vegetables to preserve colour and texture.',
        'A roux is cooked fat and flour used to thicken sauces.',
        'Salt draws moisture from vegetables through osmosis.',
        'Tempering chocolate requires precise temperature control of cocoa butter.',
        'Sous-vide seals ingredients in vacuum bags before water-bath cooking.',
        'Yeast leavens bread by producing carbon dioxide during fermentation.',
        'Caramelisation breaks down sugar into complex flavour compounds.',
        'Acids denature proteins without heat, as in ceviche preparation.',
    ],
    'law': [
        'Habeas corpus protects individuals from unlawful detention.',
        'Tort law addresses civil wrongs that cause harm to individuals.',
        'Judicial review allows courts to assess executive action legality.',
        'Mens rea is the mental element required for criminal liability.',
        'Contract formation requires offer, acceptance, and consideration.',
        'Stare decisis binds lower courts to precedent from higher courts.',
        'Equity developed to correct the rigidities of common law.',
        'Statutory interpretation gives courts power to apply legislative intent.',
        'Due process guarantees fair treatment through the judicial system.',
        'Negligence requires a duty of care, breach, and resulting damage.',
        'Property rights determine who may use and exclude others from land.',
        'International treaties bind signatory states under public international law.',
    ],
    'music': [
        'Counterpoint layers independent melodic lines in harmonious interaction.',
        'The circle of fifths maps tonal relationships between major keys.',
        'Syncopation places rhythmic stress on normally weak beats.',
        'Timbre distinguishes two instruments playing the same pitch.',
        'Modal scales extend beyond the major and natural minor modes.',
        'Polyphony combines multiple simultaneous independent voices.',
        'Dynamics in performance range from pianissimo to fortissimo.',
        'Chord inversions alter the bass note without changing the chord quality.',
        'A fermata instructs a performer to hold a note beyond its written value.',
        'Serialism organises pitch, rhythm, and dynamics into ordered series.',
        'The Doppler effect influences pitch perception of a moving sound source.',
        'Microtonal music uses intervals smaller than a semitone.',
    ],
    'medicine': [
        'Pharmacokinetics describes how the body absorbs and eliminates drugs.',
        'The blood-brain barrier restricts passage of most molecules into the CNS.',
        'Autoimmune diseases occur when the immune system attacks the body.',
        'MRI uses magnetic fields and radio waves to produce tissue images.',
        'Homeostasis maintains internal physiological variables within narrow bounds.',
        'Apoptosis is programmed cell death triggered by internal or external signals.',
        'Antibiotic resistance arises through natural selection in bacterial populations.',
        'Synaptic transmission relays signals across the neural junction via neurotransmitters.',
        'The lymphatic system drains interstitial fluid and supports immune function.',
        'Cancer occurs when cells proliferate uncontrollably and invade surrounding tissue.',
        'Vaccination trains the immune system using attenuated or inactivated antigens.',
        'Epigenetic modifications alter gene expression without changing DNA sequence.',
    ],
    'economics': [
        'Comparative advantage explains why nations specialise and trade.',
        'Price elasticity measures how quantity demanded responds to price changes.',
        'Game theory analyses strategic interactions between rational agents.',
        'Keynesian economics advocates fiscal stimulus to counter recessions.',
        'The efficient market hypothesis states that prices reflect all available information.',
        'Externalities are costs or benefits imposed on third parties outside a transaction.',
        'Monetary policy controls money supply and interest rates to manage inflation.',
        'Public goods are non-excludable and non-rival in consumption.',
        'The Gini coefficient measures income inequality within a population.',
        'Opportunity cost is the value of the next best foregone alternative.',
        'Stagflation combines high inflation with stagnant economic growth.',
        'Moral hazard arises when insulation from risk changes behaviour.',
    ],
    'architecture': [
        'The cantilever extends a structural element beyond its support point.',
        'Post-and-lintel construction spans openings with a horizontal beam.',
        'Brutalism emphasises raw concrete and functional honesty of form.',
        'A flying buttress transfers roof thrust to an outer pier.',
        'Fenestration refers to the design and placement of windows in a building.',
        'Thermal mass absorbs heat during the day and releases it at night.',
        'The golden ratio has historically informed proportional design decisions.',
        'Parametric design uses algorithms to generate complex geometric forms.',
        'Load-bearing walls carry structural weight directly to the foundations.',
        'Green roofs reduce urban heat island effects and manage stormwater.',
        'The Doric, Ionic, and Corinthian orders define classical column styles.',
        'Adaptive reuse repurposes existing buildings for new functions.',
    ],
    'ecology': [
        'Keystone species have disproportionate effects on ecosystem structure.',
        'Nutrient cycling moves carbon, nitrogen, and phosphorus through ecosystems.',
        'Trophic cascades propagate predator impacts through food webs.',
        'Biodiversity is positively correlated with ecosystem resilience.',
        'Island biogeography predicts species richness from area and isolation.',
        'Mutualism benefits both interacting species, as in mycorrhizal networks.',
        'Succession describes directional change in community composition over time.',
        'Carrying capacity limits population size relative to available resources.',
        'Habitat fragmentation reduces connectivity and increases extinction risk.',
        'Primary productivity is the rate of biomass generation by autotrophs.',
        'Invasive species disrupt established ecological relationships.',
        'Climate change shifts species ranges toward the poles and higher altitudes.',
    ],
}

# Polysemous probes — boundary-condition test sentences
PROBES = [
    ('probe_astro_eco',   'Stars form when gravitational collapse overcomes gas pressure in a nebula.'),
    ('probe_cook_chem',   'Enzymatic browning oxidises phenolic compounds in cut fruit surfaces.'),
    ('probe_law_econ',    'Antitrust law prevents monopolistic practices that distort competitive markets.'),
    ('probe_music_phys',  'Resonance amplifies frequencies matching the natural period of a vibrating body.'),
    ('probe_med_eco',     'Zoonotic diseases transfer between animal reservoirs and human populations.'),
    ('probe_arch_eco',    'Biophilic design integrates natural elements to improve occupant wellbeing.'),
]

# Flatten corpus
all_sentences, all_labels, field_indices = [], [], {}
for f in FIELDS:
    start = len(all_sentences)
    all_sentences.extend(SENTENCES[f])
    all_labels.extend([f] * len(SENTENCES[f]))
    field_indices[f] = list(range(start, start + len(SENTENCES[f])))

N = len(all_sentences)
print(f'Corpus: {N} sentences across {len(FIELDS)} fields')
print('Field sizes:', {f: len(SENTENCES[f]) for f in FIELDS})

Corpus: 96 sentences across 8 fields
Field sizes: {'astronomy': 12, 'cooking': 12, 'law': 12, 'music': 12, 'medicine': 12, 'economics': 12, 'architecture': 12, 'ecology': 12}


## 3. Embed corpus

Uses `sentence-transformers/all-MiniLM-L6-v2` (same model as NB02/NB03/NB04).  
Falls back to random unit vectors for offline environments.

In [20]:
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embs  = model.encode(all_sentences, normalize_embeddings=True,
                         show_progress_bar=True)
    probe_embs = model.encode([s for _, s in PROBES],
                              normalize_embeddings=True)
    print(f'Embeddings shape: {embs.shape}')
    OFFLINE = False
except Exception as e:
    print(f'SentenceTransformer unavailable ({e}) — using random unit vectors')
    DIM = 384
    raw = np.random.randn(N, DIM)
    embs = raw / np.linalg.norm(raw, axis=1, keepdims=True)
    raw_p = np.random.randn(len(PROBES), DIM)
    probe_embs = raw_p / np.linalg.norm(raw_p, axis=1, keepdims=True)
    OFFLINE = True

print('Embedding matrix:', embs.shape)

Batches: 100%|██████████| 3/3 [00:00<00:00, 70.58it/s]

Embeddings shape: (96, 384)
Embedding matrix: (96, 384)


## 4. PCA-2D projection for visualisation

In [21]:
pca = PCA(n_components=2, random_state=SEED)
coords_2d  = pca.fit_transform(embs)
probe_2d   = pca.transform(probe_embs)
var_exp    = pca.explained_variance_ratio_ * 100
print(f'PCA variance explained: PC1={var_exp[0]:.1f}%  PC2={var_exp[1]:.1f}%')

PCA variance explained: PC1=5.6%  PC2=5.3%


## 5. λ-Native Seed-Absorb Algorithm

```
seeds  = { field_f : [first_sentence_of_f] }   # one seed per field
bag    = all other sentences

repeat:
    absorbed_this_round = 0
    for sentence in bag:
        lambdas = [subspace_lambda(sentence, subspace[f]) for f in fields]
        min_λ   = min(lambdas)
        best_f  = argmin
        if min_λ ≤ θ:
            attach sentence to subspace[best_f]
            absorbed_this_round += 1
until absorbed_this_round == 0 or max_rounds exceeded

boundary = { s : fields where λ(s, subspace[f]) ≤ θ }
           filtered to |fields| ≥ 2
```

In [22]:
def seed_absorb(
    embeddings: np.ndarray,
    labels: list,
    fields: list,
    field_indices: dict,
    theta: float = LAMBDA_THRESHOLD,
    alpha: float = ALPHA,
    max_rounds: int = MAX_ROUNDS,
    verbose: bool = True,
) -> dict:
    """
    Returns a result dict with:
      subspaces  : {field: list of sentence indices}
      unassigned : list of sentence indices never absorbed
      boundary   : {idx: list of fields where λ ≤ theta}
      rounds_log : list of (round, absorbed) tuples
    """
    # Seed: first sentence of each field
    subspaces  = {f: [field_indices[f][0]] for f in fields}
    bag        = [i for f in fields for i in field_indices[f][1:]]
    np.random.shuffle(bag)
    rounds_log = []

    for rnd in range(1, max_rounds + 1):
        absorbed_this_round = []
        remaining_bag       = []

        for idx in bag:
            q_vec = embeddings[idx]
            lam_per_field = {
                f: subspace_lambda(q_vec, embeddings[subspaces[f]], alpha)
                for f in fields
            }
            min_lam  = min(lam_per_field.values())
            best_f   = min(lam_per_field, key=lam_per_field.get)

            if min_lam <= theta:
                subspaces[best_f].append(idx)
                absorbed_this_round.append(idx)
            else:
                remaining_bag.append(idx)

        n_abs = len(absorbed_this_round)
        rounds_log.append((rnd, n_abs))
        bag = remaining_bag
        if verbose:
            print(f'  Round {rnd:2d}: absorbed {n_abs:3d}  |  remaining {len(bag):3d}')
        if n_abs == 0:
            break

    # --- Boundary detection ---
    # For every absorbed sentence, compute λ against all subspaces
    all_assigned = [i for f in fields for i in subspaces[f]]
    boundary = {}
    for idx in all_assigned:
        q_vec       = embeddings[idx]
        close_fields = [
            f for f in fields
            if subspace_lambda(q_vec, embeddings[subspaces[f]], alpha) <= theta
        ]
        if len(close_fields) >= 2:
            boundary[idx] = close_fields

    return dict(
        subspaces  = subspaces,
        unassigned = bag,
        boundary   = boundary,
        rounds_log = rounds_log,
    )


print(f'Running seed-absorb with θ = {LAMBDA_THRESHOLD} …')
result = seed_absorb(
    embs, all_labels, FIELDS, field_indices,
    theta=LAMBDA_THRESHOLD, verbose=True
)
print(f'\nSubspace sizes: { {f: len(result["subspaces"][f]) for f in FIELDS} }')
print(f'Unassigned:  {len(result["unassigned"])}')
print(f'Boundary:    {len(result["boundary"])}')

Running seed-absorb with θ = 0.45 …
  Round  1: absorbed  55  |  remaining  33
  Round  2: absorbed   0  |  remaining  33

Subspace sizes: {'astronomy': 3, 'cooking': 4, 'law': 8, 'music': 4, 'medicine': 7, 'economics': 4, 'architecture': 7, 'ecology': 26}
Unassigned:  33
Boundary:    56


## 5.1 Assignment and purity

In [23]:
# Build assignment vector
assigned_field = ['unassigned'] * N
for f in FIELDS:
    for idx in result['subspaces'][f]:
        assigned_field[idx] = f

df = pd.DataFrame({
    'idx':        range(N),
    'sentence':   all_sentences,
    'true_field': all_labels,
    'assigned':   assigned_field,
    'pc1':        coords_2d[:, 0],
    'pc2':        coords_2d[:, 1],
    'is_seed':    [i in [field_indices[f][0] for f in FIELDS] for i in range(N)],
    'is_boundary': [i in result['boundary'] for i in range(N)],
})

# Purity: correct field assignments among assigned sentences
assigned_mask = df['assigned'] != 'unassigned'
purity = (df.loc[assigned_mask, 'true_field'] ==
          df.loc[assigned_mask, 'assigned']).mean()

print(f'H1 — Field purity (assigned only): {purity:.3f}  (NB04 KMeans baseline Rank@1 = 0.125)')
print(f'Coverage (assigned / total):       {assigned_mask.sum()}/{N} = {assigned_mask.mean():.3f}')
print(f'Boundary sentences:                {df.is_boundary.sum()}')
print()
print(df.groupby(['true_field', 'assigned']).size().rename('count').reset_index()
        .pivot_table(index='true_field', columns='assigned', values='count', fill_value=0))

H1 — Field purity (assigned only): 0.270  (NB04 KMeans baseline Rank@1 = 0.125)
Coverage (assigned / total):       63/96 = 0.656
Boundary sentences:                56

assigned      architecture  astronomy  cooking  ecology  economics  law  \
true_field                                                                
architecture           2.0        0.0      1.0      3.0        0.0  2.0   
astronomy              1.0        2.0      1.0      3.0        0.0  0.0   
cooking                0.0        0.0      1.0      2.0        2.0  1.0   
ecology                0.0        0.0      1.0      5.0        1.0  0.0   
economics              2.0        1.0      0.0      2.0        1.0  1.0   
law                    0.0        0.0      0.0      3.0        0.0  2.0   
medicine               2.0        0.0      0.0      6.0        0.0  2.0   
music                  0.0        0.0      0.0      2.0        0.0  0.0   

assigned      medicine  music  unassigned  
true_field                           

## 6. Boundary sentence inspection (H2)

Boundary sentences are those with λ ≤ θ against two or more subspaces.  
We expect these to be semantically ambiguous or cross-field in meaning.

In [24]:
print('=== Boundary sentences ===')
for idx, fields_close in sorted(result['boundary'].items()):
    sentence    = all_sentences[idx]
    true_f      = all_labels[idx]
    assigned_f  = assigned_field[idx]
    print(f'[{idx:3d}] true={true_f:<14s} assigned={assigned_f:<14s} ')
    print(f'       close subspaces: {fields_close}')
    print(f'       "{sentence}"')
    print()

# Compute λ for polysemous probes against all subspaces
print('=== Polysemous probe λ-profiles ===')
for (p_name, p_sent), p_vec in zip(PROBES, probe_embs):
    lam_row = {f: subspace_lambda(p_vec, embs[result['subspaces'][f]]) for f in FIELDS}
    sorted_lam = sorted(lam_row.items(), key=lambda x: x[1])
    close = [f for f, v in sorted_lam if v <= LAMBDA_THRESHOLD]
    print(f'{p_name}:')
    print(f'  "{p_sent}"')
    print(f'  λ-profile: { {f: round(v, 3) for f, v in sorted_lam} }')
    print(f'  Below θ: {close}')
    print()

=== Boundary sentences ===
[  0] true=astronomy      assigned=astronomy      
       close subspaces: ['cooking', 'law', 'music', 'medicine', 'economics', 'architecture', 'ecology']
       "The luminosity of a main-sequence star scales with its mass."

[  1] true=astronomy      assigned=medicine       
       close subspaces: ['cooking', 'law', 'medicine', 'architecture', 'ecology']
       "Neutron stars rotate at hundreds of revolutions per second."

[  2] true=astronomy      assigned=ecology        
       close subspaces: ['cooking', 'law', 'medicine', 'architecture', 'ecology']
       "Dark matter does not emit or absorb electromagnetic radiation."

[  5] true=astronomy      assigned=architecture   
       close subspaces: ['cooking', 'law', 'medicine', 'architecture', 'ecology']
       "Binary star systems can transfer mass between their components."

[  7] true=astronomy      assigned=ecology        
       close subspaces: ['cooking', 'law', 'medicine', 'architecture', 'ecology'

## 7. Per-subspace λ-fingerprints and Wasserstein matrix (H3)

In [ ]:
# Collect per-field λ distributions from the discovered subspaces
fingerprints = {}
for f in FIELDS:
    idxs = result['subspaces'][f]
    if len(idxs) < 2:
        fingerprints[f] = np.array([0.0])
        continue
    lambdas = build_lambda_map(embs[idxs])[0]
    fingerprints[f] = lambdas
    print(f'{f:<14s}  n={len(idxs):3d}  mean_λ={lambdas.mean():.4f}  std_λ={lambdas.std():.4f}')

# Wasserstein distance matrix
nf = len(FIELDS)
W_matrix = np.zeros((nf, nf))
for i, fi in enumerate(FIELDS):
    for j, fj in enumerate(FIELDS):
        if i != j:
            W_matrix[i, j] = wasserstein_distance(fingerprints[fi], fingerprints[fj])

mean_off_diag = W_matrix[W_matrix > 0].mean()
print(f'\nH3 — Mean off-diagonal Wasserstein: {mean_off_diag:.4f}  (NB04 KMeans baseline ≈ 0.07)')

W_df = pd.DataFrame(W_matrix, index=FIELDS, columns=FIELDS)
print(W_df.round(4))

AttributeError: 'tuple' object has no attribute 'mean'

## 8. Threshold sweep — H4

In [ ]:
thresholds = np.arange(0.20, 0.66, 0.05)
sweep_results = []

for theta in thresholds:
    r = seed_absorb(embs, all_labels, FIELDS, field_indices,
                    theta=theta, verbose=False)
    assigned_f = ['unassigned'] * N
    for f in FIELDS:
        for idx in r['subspaces'][f]:
            assigned_f[idx] = f
    assigned_mask = [a != 'unassigned' for a in assigned_f]
    n_assigned  = sum(assigned_mask)
    n_correct   = sum(all_labels[i] == assigned_f[i]
                      for i in range(N) if assigned_mask[i])
    purity_t    = n_correct / n_assigned if n_assigned > 0 else 0.0
    sweep_results.append(dict(
        theta     = round(float(theta), 2),
        purity    = purity_t,
        coverage  = n_assigned / N,
        n_boundary= len(r['boundary']),
        n_unassigned = N - n_assigned,
    ))
    print(f'θ={theta:.2f}  purity={purity_t:.3f}  coverage={n_assigned/N:.3f}  boundary={len(r["boundary"])}')

sweep_df = pd.DataFrame(sweep_results)
sweep_df.to_csv(OUTPUT_DIR / 'threshold_sweep.csv', index=False)
print('\nSaved threshold_sweep.csv')

## 9. Charts

### Chart 1 — PCA-2D scatter coloured by discovered subspace

In [ ]:
PALETTE = px.colors.qualitative.Dark24
field_color = {f: PALETTE[i % len(PALETTE)] for i, f in enumerate(FIELDS)}
field_color['unassigned'] = '#cccccc'

fig1 = go.Figure()

for f in FIELDS + ['unassigned']:
    mask = (df['assigned'] == f) & ~df['is_seed'] & ~df['is_boundary']
    sub  = df[mask]
    fig1.add_trace(go.Scatter(
        x=sub.pc1, y=sub.pc2,
        mode='markers',
        marker=dict(size=7, color=field_color[f], opacity=0.75),
        name=f,
        customdata=sub[['sentence', 'true_field']],
        hovertemplate='<b>%{customdata[1]}</b><br>%{customdata[0]}<extra></extra>',
    ))

# Seeds
seeds_df = df[df.is_seed]
fig1.add_trace(go.Scatter(
    x=seeds_df.pc1, y=seeds_df.pc2,
    mode='markers+text',
    marker=dict(size=14, symbol='diamond', color=[field_color[f] for f in seeds_df.true_field],
                line=dict(color='black', width=1.5)),
    text=seeds_df.true_field.str[:4], textposition='top center',
    name='Seed', showlegend=True,
))

fig1.update_layout(
    title='Chart 1 — λ-native subspace assignments (PCA-2D)',
    xaxis_title=f'PC1 ({var_exp[0]:.1f}%)',
    yaxis_title=f'PC2 ({var_exp[1]:.1f}%)',
    width=900, height=620,
    legend_title='Subspace',
)
fig1.write_image(str(OUTPUT_DIR / 'chart1_subspace_scatter.png'), scale=2)
fig1.show()
print('Chart 1 saved.')

### Chart 2 — Boundary sentences

In [ ]:
fig2 = go.Figure()

# Background: all non-boundary points, muted
mask_bg = ~df.is_boundary & ~df.is_seed
fig2.add_trace(go.Scatter(
    x=df.loc[mask_bg, 'pc1'], y=df.loc[mask_bg, 'pc2'],
    mode='markers',
    marker=dict(size=5, color=[field_color.get(f, '#ccc') for f in df.loc[mask_bg, 'assigned']],
                opacity=0.35),
    name='Interior',
    customdata=df.loc[mask_bg, ['sentence', 'true_field', 'assigned']],
    hovertemplate='true=%{customdata[1]} assigned=%{customdata[2]}<br>%{customdata[0]}<extra></extra>',
))

# Boundary points
bound_df = df[df.is_boundary]
bound_labels = [', '.join(result['boundary'][idx]) for idx in bound_df['idx']]
fig2.add_trace(go.Scatter(
    x=bound_df.pc1, y=bound_df.pc2,
    mode='markers+text',
    marker=dict(size=13, symbol='star', color='crimson',
                line=dict(color='black', width=1)),
    text=[str(i) for i in bound_df['idx']],
    textposition='top center',
    name='Boundary (★)',
    customdata=list(zip(bound_df['sentence'], bound_df['true_field'], bound_labels)),
    hovertemplate='true=%{customdata[1]}<br>close=%{customdata[2]}<br>%{customdata[0]}<extra></extra>',
))

# Probes
p_close_labels = []
for p_vec in probe_embs:
    close = [f for f in FIELDS
             if subspace_lambda(p_vec, embs[result['subspaces'][f]]) <= LAMBDA_THRESHOLD]
    p_close_labels.append(', '.join(close) if close else 'none')
fig2.add_trace(go.Scatter(
    x=probe_2d[:, 0], y=probe_2d[:, 1],
    mode='markers+text',
    marker=dict(size=12, symbol='cross', color='orange',
                line=dict(color='black', width=1)),
    text=[n for n, _ in PROBES],
    textposition='bottom center',
    name='Probe',
    customdata=list(zip([s for _, s in PROBES], p_close_labels)),
    hovertemplate='close=%{customdata[1]}<br>%{customdata[0]}<extra></extra>',
))

fig2.update_layout(
    title='Chart 2 — Boundary sentences (★) and polysemous probes (✕)',
    xaxis_title=f'PC1 ({var_exp[0]:.1f}%)',
    yaxis_title=f'PC2 ({var_exp[1]:.1f}%)',
    width=900, height=620,
)
fig2.write_image(str(OUTPUT_DIR / 'chart2_boundary_scatter.png'), scale=2)
fig2.show()
print('Chart 2 saved.')

### Chart 3 — Threshold sweep

In [ ]:
fig3 = make_subplots(rows=1, cols=3,
    subplot_titles=['Purity', 'Coverage', 'Boundary count'])

for col, metric, label in [
    (1, 'purity',     'Field purity'),
    (2, 'coverage',   'Coverage (assigned / N)'),
    (3, 'n_boundary', 'Boundary sentences'),
]:
    fig3.add_trace(
        go.Scatter(x=sweep_df.theta, y=sweep_df[metric],
                   mode='lines+markers',
                   name=label, line=dict(width=2)),
        row=1, col=col
    )
    fig3.add_vline(x=LAMBDA_THRESHOLD, line_dash='dash', line_color='red',
                   annotation_text='θ=0.45', row=1, col=col)

fig3.update_layout(
    title='Chart 3 — Threshold sweep (H4): purity, coverage, boundary count vs θ',
    width=1200, height=400,
    showlegend=False,
)
fig3.write_image(str(OUTPUT_DIR / 'chart3_threshold_sweep.png'), scale=2)
fig3.show()
print('Chart 3 saved.')

### Chart 4 — Per-subspace λ-fingerprint gallery

In [ ]:
nf   = len(FIELDS)
cols = 4
rows = int(np.ceil(nf / cols))
fig4 = make_subplots(rows=rows, cols=cols,
                     subplot_titles=FIELDS)

for i, f in enumerate(FIELDS):
    r_i = i // cols + 1
    c_i = i % cols + 1
    lams = fingerprints[f]
    fig4.add_trace(
        go.Histogram(x=lams, nbinsx=12,
                     marker_color=field_color[f],
                     name=f, showlegend=False,
                     hovertemplate='λ=%{x:.3f}<extra></extra>'),
        row=r_i, col=c_i
    )
    fig4.add_vline(x=LAMBDA_THRESHOLD, line_dash='dash', line_color='red',
                   row=r_i, col=c_i)

fig4.update_layout(
    title='Chart 4 — Per-subspace λ-fingerprint gallery (red line = θ)',
    width=1100, height=480,
)
fig4.write_image(str(OUTPUT_DIR / 'chart4_fingerprint_gallery.png'), scale=2)
fig4.show()
print('Chart 4 saved.')

### Chart 5 — Wasserstein distance matrix

In [ ]:
fig5 = go.Figure(go.Heatmap(
    z=W_matrix,
    x=FIELDS, y=FIELDS,
    colorscale='Blues',
    text=np.round(W_matrix, 3),
    texttemplate='%{text}',
    colorbar_title='W₁',
))
fig5.update_layout(
    title=f'Chart 5 — Wasserstein-1 matrix (λ-native subspaces)  mean off-diag={mean_off_diag:.4f}',
    width=720, height=600,
    xaxis_title='Subspace', yaxis_title='Subspace',
)
fig5.write_image(str(OUTPUT_DIR / 'chart5_wasserstein_matrix.png'), scale=2)
fig5.show()
print('Chart 5 saved.')

## 10. Hypothesis summary

In [ ]:
print('=== NB05 Hypothesis Digest ===')
print()

# H1
nb04_baseline_rank1 = 0.125
h1_verdict = 'STRONG' if purity > nb04_baseline_rank1 * 2 else \
             'WEAK'   if purity > nb04_baseline_rank1 else 'FAILED'
print(f'H1 — Purity > NB04 KMeans Rank@1')
print(f'     λ-native purity: {purity:.3f}   NB04 baseline: {nb04_baseline_rank1}')
print(f'     Verdict: {h1_verdict}')
print()

# H2
n_boundary = len(result['boundary'])
print(f'H2 — Boundary sentences are semantically ambiguous')
print(f'     Found {n_boundary} boundary sentences (inspect section 6 for qualitative check)')
print(f'     Verdict: QUALITATIVE — see boundary table above')
print()

# H3
nb04_baseline_w = 0.07
h3_verdict = 'STRONG' if mean_off_diag > nb04_baseline_w * 1.2 else \
             'WEAK'   if mean_off_diag > nb04_baseline_w        else 'FAILED'
print(f'H3 — Off-diagonal Wasserstein > NB04 KMeans baseline')
print(f'     λ-native mean W: {mean_off_diag:.4f}   NB04 baseline: {nb04_baseline_w}')
print(f'     Verdict: {h3_verdict}')
print()

# H4 — purity peaks at θ near 0.45?
best_row = sweep_df.loc[sweep_df.purity.idxmax()]
h4_verdict = 'STRONG' if abs(best_row.theta - 0.45) <= 0.10 else 'WEAK'
print(f'H4 — θ = 0.45 is near the purity peak')
print(f'     Peak purity {best_row.purity:.3f} at θ = {best_row.theta}')
print(f'     Verdict: {h4_verdict}')
print()

print('=== End of NB05 ===')

## 11. Series notes and next steps

| NB | Key contribution |
|---|---|
| NB01 | Synthetic baseline: ArrowSpace improves local-minima quality |
| NB02 | Real embeddings: per-word λ-maps, polysemous probes |
| NB03 | Sentence corpus: BasinHop + ArrowSpace, α-sweep, boundary sentences |
| NB04 | λ-fingerprints: field distinctiveness, KMeans/RP tiling, H1–H4 |
| **NB05** | **λ-native subspace clustering: seed-absorb, boundary detection, threshold sweep** |
| NB06 (planned) | Cross-layer λ-fingerprint tracking — at which transformer layer do fields crystallise? |
| NB07 (planned) | Causal attribution: link λ-fingerprints to specific attention heads and MLP neurons |

### Output files
```
output__05/
  chart1_subspace_scatter.png
  chart2_boundary_scatter.png
  chart3_threshold_sweep.png
  chart4_fingerprint_gallery.png
  chart5_wasserstein_matrix.png
  threshold_sweep.csv
```

### Open questions for NB06
- Does the crystallisation layer (where λ-fingerprints become distinct) differ across fields?
- Are boundary sentences consistently boundary across transformer layers, or only at the output layer?
- Can θ = 0.45 be derived from the spectrum of the Laplacian (e.g. spectral gap) rather than fixed?
